# 05_model_comparison

Compare all models: metrics, ROC and precision-recall curves.

---

*DiaPredict AI — Explainable Diabetes Risk Prediction System.* *Educational / research prototype, not a medical diagnosis.*

## Objectives
- Compare all four models on the same test set
- Plot ROC curves, precision-recall curves and confusion matrices
- Identify the best-performing model

In [ ]:
import sys
from pathlib import Path

# Make the project root importable no matter where the notebook runs.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Root:", PROJECT_ROOT)


### 1. Prepare data and load trained models

In [ ]:
from backend.app.preprocessing import load_raw_dataset, make_train_test
from backend.app.inference import ml_predict_proba
from backend.app.model_loader import get_registry

raw = load_raw_dataset()
X_train, y_train, X_test, y_test, scaler, medians = make_train_test(raw)

registry = get_registry()
print([m.model_id for m in registry.loaded()])

### 2. Collect predictions and metrics

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

probs, labels, names = [], [], []
for m in registry.loaded():
    if m.kind == "dl":
        import torch
        x = torch.tensor(X_test, dtype=torch.float32)
        m.model.eval()
        with torch.no_grad():
            p = m.model(x).numpy()
    else:
        p = m.model.predict_proba(X_test)[:, 1]
    probs.append(p); labels.append(m.model_name); names.append(m.model_id)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
rows = []
for name, p in zip(names, probs):
    pred = (p >= 0.5).astype(int)
    rows.append({
        "model_id": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, p),
    })
pd.DataFrame(rows).set_index("model_id").round(3)

### 3. ROC curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, p in zip(labels, probs):
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves — held-out test set"); plt.legend(); plt.tight_layout(); plt.show()

### 4. Precision-recall curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, p in zip(labels, probs):
    prec, rec, _ = precision_recall_curve(y_test, p)
    plt.plot(rec, prec, label=f"{name} (AP={average_precision_score(y_test, p):.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision-recall curves"); plt.legend(); plt.tight_layout(); plt.show()

## Interpretation

The ROC-AUC and precision-recall curves let us compare models independently of an arbitrary threshold. Because the dataset is imbalanced, the PR curve is the more informative view of the positive class. The chosen "best" model is the one with the best trade-off for a given use case — in our comparison this is typically Random Forest by F1 and Logistic Regression by ROC-AUC.